# Bonus Challenge Five: Deploying Agents to Agent Platform (Agent Engine)
### Agentic AI with the Google Agent Development Kit (ADK) — Skills Validation Workshop (10 Points)

**Objectives:**
- Demonstrate the ability to deploy and run an intelligent agent on **Google Cloud Agent Platform** (Vertex AI Agent Engine).
- Complete the 4-step deployment lifecycle:
  1. **Initialize Vertex AI** with project, location, and Cloud Storage staging bucket settings.
  2. **Define an Agent** using the Google Agent Development Kit (ADK) equipped with tools (e.g. `google_search`).
  3. **Test the Agent Locally** using Vertex AI `reasoning_engines.AdkApp`.
  4. **Deploy to Agent Platform** using `vertexai.agent_engines.create(...)` and test the live remote endpoint with `stream_query`.

## 1. Install Dependencies
Install the required Vertex AI Agent Engine and ADK extensions.

In [1]:
!pip install --upgrade --quiet \
    "google-cloud-aiplatform[agent_engines,adk]" \
    google-adk \
    requests \
    googlemaps

## 2. Google Cloud Configuration & Vertex AI Initialization
Configure project information, region, and GCS staging bucket.

In [2]:
import os
import google.auth
import vertexai

# 1. Resolve Project ID from environment or gcloud config
try:
    _, project_id = google.auth.default()
except Exception:
    project_id = None

# Workshop Project Settings
PROJECT_ID = project_id or os.getenv("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-01-7a38d08f6f78")
LOCATION = "us-central1" # @param {type:"string"}
MODEL_LOCATION = "global" # @param {type:"string"}
STAGING_BUCKET = f"gs://agent-engine-staging-{PROJECT_ID}" # @param {type:"string"}

# Agent Runtime is regional. AdkApp captures this location for sessions.
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

print(f"[Vertex AI Initialized]")
print(f"  Project ID:     {PROJECT_ID}")
print(f"  Region:         {LOCATION}")
print(f"  Staging Bucket: {STAGING_BUCKET}")

[Vertex AI Initialized]
  Project ID:     qwiklabs-gcp-01-7a38d08f6f78
  Region:         us-central1
  Staging Bucket: gs://agent-engine-staging-qwiklabs-gcp-01-7a38d08f6f78


## 3. Define the Agent using Google ADK
Per Slide 64, create an agent equipped with the built-in `google_search` tool.

In [3]:
from google.adk.tools import google_search
from google.adk.agents import Agent

search_agent = Agent(
    name="Google",
    model="gemini-2.5-flash",
    description="You can search Google and return real-time answers from the Internet.",
    instruction=(
        """You are a helpful Chatbot deployed on Google Cloud Agent Platform.
        If you need up-to-date facts, news, sports scores, or information you don't know, search Google for the answer."""
    ),
    tools=[google_search],
)

print(f"[Agent Created] {search_agent.name} (Model: {search_agent.model})")
print(f"  Tools: {[type(t).__name__ for t in search_agent.tools]}")

[Agent Created] Google (Model: gemini-2.5-flash)
  Tools: ['GoogleSearchTool']


## 4. Test the Agent Locally
Per Slide 65, wrap the agent inside `AdkApp` and run a local verification query before deploying.

In [4]:
from IPython.display import Markdown, display
from vertexai.preview.reasoning_engines import AdkApp

app = AdkApp(
    agent=search_agent,
    env_vars={"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION},
)

print("Running local query via AdkApp...")
local_response = ""
for event in app.stream_query(
    user_id="USER_ID",
    message="Give me the latest news highlights in the world of science and technology.",
):
    if "content" in event and event["content"].get("parts"):
        for part in event["content"]["parts"]:
            if "text" in part:
                local_response += part["text"]

display(Markdown(local_response if local_response else "(Local test stream completed)"))

Running local query via AdkApp...


/usr/local/lib/python3.12/dist-packages/vertexai/preview/reasoning_engines/templates/adk.py:1138: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  self._tmpl_attrs["credential_service"] = InMemoryCredentialService()
/usr/local/lib/python3.12/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()


Here are the latest news highlights in the world of science and technology:

**Technology:**

*   **Artificial Intelligence (AI) Dominance and Concerns:** AI continues to be a major focus, with discussions around companies capping AI spending and securing mission-critical enterprise AI architectures using isolated Google Cloud systems for data sovereignty. Security concerns are also prominent, with banks restricting AI agents due to customer account security, and warnings about potential AI lawsuits and global outages by 2027. There's an increasing emphasis on pre-execution security for AI-generated code. Apple's new iMac with M6 chip promises a significant leap in AI performance, and the company has integrated AI into various products, including updates to Final Cut.
*   **New Devices and Product Innovations:** Apple has been active with new releases, including updated iPads and MacBooks, a new HomeView hub expanding its smart home strategy, and a new HomePod mini 2. Vivo launched its X500 Pro Max, a camera-focused smartphone, and the iPhone Duo has received updates for its camera app, Halide. Lenovo also introduced a student-ready Idea Tab bundle.
*   **Company Strategies and Digital Transformation:** Disney appointed its first Chief Technology Officer to advance its tech strategy. Zoom is reportedly planning to bridge the gap between chat and execution, and NVIDIA is making infrastructure upgrades for AI.
*   **Cybersecurity:** Decades' worth of flaws have been uncovered in web code, and there are internet-wide threats with security teams scrambling to patch PHP.

**Science:**

*   **Astronomy and Space Exploration:** The James Webb Space Telescope has made significant discoveries, including finding dust-rich water ice in interstellar Comet 3I/ATLAS and revealing metal-rich gas surrounding the Milky Way's central black hole. Astronomers also observed a scorching lava planet 280 light-years away that surprisingly maintains a thick atmosphere.
*   **Paleontology and Evolutionary Biology:** Recent findings include the detailed reconstruction of a Cretaceous bird and the discovery of an armored amphibian from the Triassic Period that lived entirely in the sea. A single vertebra hints at the oldest known Iberian dinosaur.
*   **Health, Medicine, and Biology:** Research highlights include the Mediterranean diet's benefits for heart, brain, and healthy aging, linked to mitochondrial proteins. Scientists have discovered that switching off a protein called Mitch can make human cells burn more fat, and that certain gut bacteria are strongly linked to the rate of biological aging. Other advancements include a world-first cancer vaccine cutting melanoma recurrence risk, a drug combination for Parkinson's symptoms, and a nose spray derived from toad venom showing promise for depression. A new study found cranberry juice appears to suppress dental plaque growth.
*   **Environmental Science and Materials:** Efforts to address environmental challenges include embedded enzymes helping plastics break down faster in the deep sea and a flash heating method for recovering critical metals from e-waste. Concerns have been raised about plastic tea bags releasing microplastic particles.
*   **Physics and Quantum Technology:** Advances in quantum computing include quantum gates setting a speed record at 6.8 nanoseconds with high accuracy, and new converters that could bridge light wavelength gaps in quantum communications.

## 5. Deploy the Agent to Google Agent Platform (Agent Engine)
Per Slide 66, deploy the agent application to managed Google Cloud infrastructure using `agent_engines.create(...)`.

In [5]:
from vertexai import agent_engines

print("Starting deployment to Vertex AI Agent Platform (this will take a few minutes)...")

remote_agent = agent_engines.create(
    agent_engine=app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
    display_name="Google Search Agent Engine",
    description="Deployed ADK Search Agent on Vertex AI Agent Platform",
)

print("\n[SUCCESS] Agent successfully deployed to Agent Platform!")
print(f"Resource Name: {getattr(remote_agent, 'resource_name', 'Active')}")

INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'pydantic': '2.13.4', 'google-cloud-aiplatform': '2.6.0'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'cloudpickle==3.1.2', 'pydantic==2.13.4']


Starting deployment to Vertex AI Agent Platform (this will take a few minutes)...


INFO:vertexai.agent_engines:Creating bucket agent-engine-staging-qwiklabs-gcp-01-7a38d08f6f78 in location='us-central1'
INFO:vertexai.agent_engines:Wrote to gs://agent-engine-staging-qwiklabs-gcp-01-7a38d08f6f78/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://agent-engine-staging-qwiklabs-gcp-01-7a38d08f6f78/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://agent-engine-staging-qwiklabs-gcp-01-7a38d08f6f78/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/522969117335/locations/us-central1/reasoningEngines/21482533061394432/operations/7553614865294688256
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-01-7a38d08f6f78
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/5229691


[SUCCESS] Agent successfully deployed to Agent Platform!
Resource Name: projects/522969117335/locations/us-central1/reasoningEngines/21482533061394432


## 6. Test the Deployed Remote Agent Engine
Query the live, deployed agent running on Google Cloud Agent Platform.

In [6]:
print("Querying deployed remote Agent Engine...")

remote_response = ""
for event in remote_agent.stream_query(
    user_id="agent-engine-test-user",
    message="Give me the news highlights in the world of sports.",
):
    print("Remote Event:", event)
    if "content" in event and event["content"].get("parts"):
        for part in event["content"]["parts"]:
            if "text" in part:
                remote_response += part["text"]

print("\n--- FINAL REMOTE AGENT RESPONSE ---")
display(Markdown(remote_response if remote_response else "(Completed)"))

Querying deployed remote Agent Engine...
Remote Event: {'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': "Here are some of the latest sports highlights from around the world:\n\n**Baseball**\nThe Milwaukee Brewers are on the verge of their first World Series title, having routed the Tampa Bay Rays 10-1 and moving to within one win. They previously came from behind to win Game 2, taking a 2-0 series lead. In Game 3, the Brewers had a historic 8-run, two-out surge.\n\n**Football (Soccer)**\nHarry Kane and Alexia Putellas have been named the winners of this year's Ballon d'Or awards. There was a brief mix-up when Harry Kane was mistakenly announced as the winner prematurely by organizers in an article published too early. Kylian Mbappé scored twice to lead Real Madrid to a 2-1 victory over Barcelona, ending their perfect league start.\n\n**American Football (NFL)**\nThe Pittsburgh Steelers defeated the New Orleans Saints 16-10 in the NFL's first-ever game in Paris. Chri

Here are some of the latest sports highlights from around the world:

**Baseball**
The Milwaukee Brewers are on the verge of their first World Series title, having routed the Tampa Bay Rays 10-1 and moving to within one win. They previously came from behind to win Game 2, taking a 2-0 series lead. In Game 3, the Brewers had a historic 8-run, two-out surge.

**Football (Soccer)**
Harry Kane and Alexia Putellas have been named the winners of this year's Ballon d'Or awards. There was a brief mix-up when Harry Kane was mistakenly announced as the winner prematurely by organizers in an article published too early. Kylian Mbappé scored twice to lead Real Madrid to a 2-1 victory over Barcelona, ending their perfect league start.

**American Football (NFL)**
The Pittsburgh Steelers defeated the New Orleans Saints 16-10 in the NFL's first-ever game in Paris. Chris Boswell made three field goals, and Aaron Rodgers threw for over 200 yards in the Steelers' victory.
In other NFL action, the Dallas Cowboys beat the Philadelphia Eagles 38-21, with Dak Prescott throwing three touchdown passes. Despite their 2-4 record, Cowboys star CeeDee Lamb believes the team can still win the NFC East. Lawrence McCutcheon, a five-time Pro Bowl running back for the Los Angeles Rams, passed away at 76. Jay-Z is set to headline the 2027 Super Bowl halftime show.

**Basketball (NBA & WNBA)**
Cleveland Cavaliers games will air on KDKA+ during the 2026-27 NBA season, with 15 games scheduled. In the WNBA, the Valkyries have pushed the Dream to the brink, needing one more win for the WNBA title.

**Other Sports**
In hockey, Leo Carlsson rallied the Anaheim Ducks to a 5-4 victory over the Philadelphia Flyers, snapping their winning streak. In gymnastics, Skye Blakely won the world all-around title, completing a comeback. Kyle Larson earned his first Talladega win and extended his NASCAR Chase points lead.

## 7. Clean Up Deployed Resources (Optional)
Uncomment and run the cell below when finished to delete the Agent Engine instance and avoid ongoing charges.

In [7]:
# To delete the deployed Agent Engine:
# remote_agent.delete(force=True)
# print("Deployed Agent Engine resource deleted.")

## 8. Summary & Verification
✅ **Agent Definition**: ADK Agent configured with Google Search tool.
✅ **Local Validation**: Tested with `AdkApp` and local query streaming.
✅ **Cloud Deployment**: Deployed to Vertex AI Agent Platform via `agent_engines.create(...)`.
✅ **Live Remote Testing**: Verified remote streaming query execution on Agent Engine.